In [2]:
from torch.func import vjp
import torch
from torch.func import jacrev, functional_call
import torch.nn as nn
from torch import Tensor


In [3]:
import sys
import os

current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '../../'))

# 将这个父目录添加到sys.path的最前面
if project_root_dir not in sys.path:
    sys.path.insert(0, project_root_dir)

print(sys.path)

['/home/hqdeng7/lijuyang/generalization', '/home/hqdeng7/.conda/envs/ljy/lib/python311.zip', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/lib-dynload', '', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/site-packages']


In [4]:
inputs = torch.randn(3)  # 输入张量
func = torch.sin      # 你的函数，这里是torch.sin
cotangents = (torch.randn(3),) # 余切向量，与func的输出形状相同

outputs, vjp_fn = vjp(func, inputs)
vjps = vjp_fn(*cotangents)

vjps, torch.cos(inputs) * cotangents[0]

((tensor([-0.3151, -0.9357,  0.5892]),), tensor([-0.3151, -0.9357,  0.5892]))

In [5]:
def compute_batch_logits(model, params, buffers, inputs):
	return functional_call(model, (params, buffers), inputs)

In [6]:
## 1. 定义你的 PyTorch 模型
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        logits = self.fc2(x) # 我们要计算关于这些 logits 的雅可比
        return logits

## 2. 实例化模型
input_dim = 10
hidden_dim = 20
output_dim = 5
model = SimpleNet(input_dim, hidden_dim, output_dim)

params = dict(model.named_parameters())
buffers = dict(model.named_buffers())
model.eval()
input0 = torch.randn(1, input_dim)
input1 = torch.randn(1, input_dim)
inputs = torch.concat((input0, input1))

res = jacrev(compute_batch_logits, argnums=1)(model, params, buffers, inputs)
res

{'fc1.weight': tensor([[[[ 5.8513e-02, -6.6115e-02,  1.8684e-02,  ...,  8.8652e-02,
            -2.8668e-01, -6.3349e-02],
           [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
             0.0000e+00,  0.0000e+00],
           [-9.5144e-02,  1.0751e-01, -3.0381e-02,  ..., -1.4415e-01,
             4.6615e-01,  1.0301e-01],
           ...,
           [-5.8970e-02,  6.6632e-02, -1.8830e-02,  ..., -8.9345e-02,
             2.8892e-01,  6.3844e-02],
           [-9.5891e-02,  1.0835e-01, -3.0619e-02,  ..., -1.4528e-01,
             4.6980e-01,  1.0382e-01],
           [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
             0.0000e+00,  0.0000e+00]],
 
          [[-2.7782e-02,  3.1392e-02, -8.8711e-03,  ..., -4.2092e-02,
             1.3611e-01,  3.0078e-02],
           [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
             0.0000e+00,  0.0000e+00],
           [ 3.3200e-02, -3.7513e-02,  1.0601e-02,  ...,  5.0300e-02,
            -1.6266e-01, 

In [7]:
res['fc1.weight'].shape

torch.Size([2, 5, 20, 10])

In [8]:
torch.concat(tuple(jacob.flatten(2) for jacob in res.values()), dim=2).shape

torch.Size([2, 5, 325])

In [9]:
def compute_batch_logits(model, params, buffers, inputs):
	return functional_call(model, (params, buffers), inputs)

def get_param_jac(model : nn.Module, params : dict, buffers : dict, 
				  inputs : Tensor):
	'''
	Parameters:  
		inputs: a batch of inputs  

	Return:  
		the jacobian matrix shape (batchsize, outputdim, paramnum)  
	'''
	param_jacob : dict = jacrev(compute_batch_logits, argnums=1)(model, params, buffers, inputs)
	return torch.concat(tuple(jacob.flatten(2) for jacob in param_jacob.values()), dim=2)

In [10]:
def get_entk(model, params, buffers, inputs0, inputs1):
	jacob0 = get_param_jac(model, params, buffers, inputs0)
	jacob1 = get_param_jac(model, params, buffers, inputs1)
	print(jacob0.shape)
	print(jacob1.shape)
	return torch.bmm(jacob0, jacob1.transpose(1, 2))

In [11]:
inputs0 = torch.concat((input0, input0))
inputs1 = torch.concat((input0, input1))
get_entk(model, params, buffers, inputs0, inputs1)

torch.Size([2, 5, 325])
torch.Size([2, 5, 325])


tensor([[[ 8.3823e+00, -1.0915e-01, -3.0941e-01,  7.2832e-01, -1.1772e-01],
         [-1.0915e-01,  8.1346e+00,  1.2997e-01,  1.2961e-01,  1.0383e+00],
         [-3.0941e-01,  1.2997e-01,  7.6735e+00,  7.4695e-02,  8.1372e-02],
         [ 7.2832e-01,  1.2961e-01,  7.4695e-02,  7.4371e+00, -1.0701e-01],
         [-1.1772e-01,  1.0383e+00,  8.1372e-02, -1.0701e-01,  7.1371e+00]],

        [[ 5.7648e+00,  1.4342e-02, -1.5535e-01,  9.9382e-02, -5.5688e-03],
         [ 1.4342e-02,  5.9104e+00,  5.5645e-02,  7.2022e-02,  2.4785e-01],
         [-1.5535e-01,  5.5645e-02,  5.7229e+00, -7.7448e-02, -6.4519e-03],
         [ 9.9382e-02,  7.2022e-02, -7.7448e-02,  5.6296e+00, -3.5144e-02],
         [-5.5688e-03,  2.4785e-01, -6.4519e-03, -3.5144e-02,  5.5681e+00]]],
       grad_fn=<BmmBackward0>)

In [12]:
import torchvision
import torchvision.transforms as transforms

def load_cifar10_data(data_path, batch_size, num_workers):
	"""
	加载 CIFAR-10 数据集并返回训练和测试 DataLoader。
	"""
	print(f"正在加载 CIFAR-10 数据集到 {data_path}...")

	# 数据预处理
	transform = transforms.Compose([
		transforms.ToTensor(),
		transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) # CIFAR-10 的均值和标准差
	])

	train_dataset = torchvision.datasets.CIFAR10(root=data_path, train=True, download=True, transform=transform)
	test_dataset = torchvision.datasets.CIFAR10(root=data_path, train=False, download=True, transform=transform)

	train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
	test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

	print("CIFAR-10 数据集加载成功！")
	return train_loader, test_loader

In [13]:
from loss_distribution.pytorch_script.visual_utils import load_cifar10_data, load_model_state_dict

In [14]:
train_dl, test_dl = load_cifar10_data('/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10', 1, 16)

正在加载 CIFAR-10 数据集到 /home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10...
CIFAR-10 数据集加载成功！


In [15]:
data_pth = '/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10'
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
testset = torchvision.datasets.CIFAR10(root=data_pth, train=False, download=True, transform=transform)

In [16]:
model_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_200.pth'
model = load_model_state_dict('cifar10', 'resnet20', 10, model_path, 'cuda')

  从字典中提取模型状态字典...
提取成功


In [17]:
dict(model.named_parameters()), dict(model.named_buffers())

({'conv1.weight': Parameter containing:
  tensor([[[[-4.6178e-02,  2.5772e-02,  1.4103e-01],
            [-2.2016e-01, -6.5458e-02,  3.2264e-01],
            [-1.2196e-01, -1.0348e-01,  1.1708e-01]],
  
           [[-1.9717e-01, -7.1301e-02,  1.9544e-01],
            [-3.0966e-01, -7.7259e-02,  4.1137e-01],
            [-1.1956e-01, -6.8082e-02,  1.2753e-01]],
  
           [[-1.8422e-01, -5.9668e-02,  1.7424e-01],
            [-2.4601e-01, -5.5391e-02,  3.4165e-01],
            [-9.3654e-02, -4.7443e-02,  1.4044e-01]]],
  
  
          [[[ 3.0607e-02,  1.8445e-01,  7.5907e-02],
            [ 2.3080e-02, -1.6054e-02,  7.9851e-02],
            [ 4.1655e-03, -1.3483e-01, -1.0361e-02]],
  
           [[-7.3002e-02, -6.6162e-02, -6.7823e-02],
            [-9.9486e-02, -2.7115e-01, -4.5764e-02],
            [-3.2883e-02, -2.3395e-01, -1.2991e-02]],
  
           [[ 2.6784e-04,  2.8533e-02, -3.0836e-02],
            [ 5.5074e-02, -2.1643e-02,  5.7356e-02],
            [ 8.8925e-02,  7.5527e-

In [18]:
from tqdm import tqdm

In [19]:

model.eval() # Set model to evaluation mode (disables dropout, batchnorm updates)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Choose a reference sample (e.g., the first sample in the test set)
reference_sample, _ = testset[0]
reference_sample = reference_sample.unsqueeze(0).to(device) # Add batch dimension and move to device

entk_results = []

print("\nCalculating 'entk' for the reference sample against all other samples...")
# Iterate through all other samples in the dataset
# We start from index 1 to compare against samples different from the reference sample
for i in range(10):
	current_sample, _ = testset[i]
	current_sample = current_sample.unsqueeze(0).to(device) # Add batch dimension and move to device

	entk_val = get_entk(model, dict(model.named_parameters()), dict(model.named_buffers()), 
					reference_sample, current_sample)
	entk_results.append(entk_val.cpu())
	print(entk_val)



Calculating 'entk' for the reference sample against all other samples...


/home/hqdeng7/.conda/envs/ljy/lib/python3.11/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


torch.Size([1, 10, 269722])
torch.Size([1, 10, 269722])
tensor([[[ 3.0445e+04, -1.7597e+03, -5.9909e+03, -1.7877e+04, -2.8769e+03,
          -4.0684e+03, -5.7248e+02, -1.9146e+03, -1.0602e+03,  5.7357e+03],
         [-1.7597e+03,  4.5157e+04, -7.7196e+03, -2.0525e+04, -4.3899e+03,
          -8.8933e+03, -3.8879e+03, -4.7467e+03,  1.1044e+03,  5.7091e+03],
         [-5.9909e+03, -7.7196e+03,  4.3574e+04, -5.3636e+03, -2.1327e+03,
          -2.3419e+04,  7.7224e+03, -4.5214e+03,  2.7463e+03, -4.8401e+03],
         [-1.7877e+04, -2.0525e+04, -5.3636e+03,  8.9823e+04, -4.8818e+03,
           1.3050e+04, -1.1085e+04, -1.2173e+04, -1.5096e+04, -1.5815e+04],
         [-2.8769e+03, -4.3899e+03, -2.1327e+03, -4.8818e+03,  1.0842e+04,
          -2.4811e+03, -6.1888e+02,  4.3895e+03,  3.1032e+01,  2.1797e+03],
         [-4.0684e+03, -8.8933e+03, -2.3419e+04,  1.3050e+04, -2.4811e+03,
           5.6127e+04, -1.3466e+04,  2.2939e+03, -1.2363e+04, -6.7206e+03],
         [-5.7248e+02, -3.8879e+03,  7

In [20]:
for entk in entk_results:
	print(torch.linalg.norm(entk))

tensor(157425.9688, grad_fn=<LinalgVectorNormBackward0>)
tensor(31275.0254, grad_fn=<LinalgVectorNormBackward0>)
tensor(41602.7617, grad_fn=<LinalgVectorNormBackward0>)
tensor(35051.4023, grad_fn=<LinalgVectorNormBackward0>)
tensor(40054.9492, grad_fn=<LinalgVectorNormBackward0>)
tensor(57624.4922, grad_fn=<LinalgVectorNormBackward0>)
tensor(39217.4961, grad_fn=<LinalgVectorNormBackward0>)
tensor(41853.7461, grad_fn=<LinalgVectorNormBackward0>)
tensor(54200.2930, grad_fn=<LinalgVectorNormBackward0>)
tensor(39934.5391, grad_fn=<LinalgVectorNormBackward0>)
